In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))  # Add parent directory to path

import numpy as np
# from DG_classes import *
import matplotlib.pyplot as plt
import plotly.figure_factory as ff
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from visualisation import *
import imageio
from scipy.sparse import diags, bmat

from methods.pde import *

# import warnings
# # Suppress specific warnings
# warnings.filterwarnings("ignore", category=DeprecationWarning)

In [26]:
# Load ABM data
import pandas as pd
import glob

# Parameters
sim_id = 45  # Change this to load different simulation

# Find all files with 0 percent noise for specific simulation
file_pattern = f'../data/ABM_Coordinates/simID{sim_id}_*_0percentNoise.csv'
files = sorted(glob.glob(file_pattern))

# Load and stack all timesteps
all_timesteps = []
for file in files:
    df = pd.read_csv(file)
    # Filter for macrophages and extract coordinates + oxygen
    macrophage_data = df[df['PointType'] == 'Macrophage'][['x', 'y', 'Oxygen']].values  # shape: (n_points, 3)
    all_timesteps.append(macrophage_data)

# Stack into [t, n, 3] array
data = np.stack(all_timesteps, axis=0)  # shape: (t, n, 3)
true_oxygen = data[:, :, 2]  # shape: (t, n)
data = data[:, :, :2]  # Keep only x and y coordinates, shape: (t, n, 2)
print(f"Loaded position data shape: {data.shape}")
print(f"Loaded oxygen data shape: {true_oxygen.shape}")

# Create visualization for the last timestep
fig = px.scatter(x=data[-1, :, 0], 
                 y=data[-1, :, 1], 
                 color=true_oxygen[-1])
fig.update_layout(width=800, height=600, title=f"Last timestep (t={len(data)-1})")
fig.show()


Loaded position data shape: (26, 100, 2)
Loaded oxygen data shape: (26, 100)


In [32]:
# Extract the vector field from positions:

# Position data at time up to t-1
x = data[:-1] # [t-1, n, 2]
x = x.reshape(-1, 2) # [N, 2] where N = (t-1) * n

# Vector field (velocity) is the difference between consecutive positions
vf = data[1:] - data[:-1] # [t-1, n, 2]
vf = vf.reshape(-1, 2) # [N, 2]

# Visualize data and vector field
fig = ff.create_quiver(x[:, 0], x[:, 1], 
                       vf[:, 0], vf[:, 1], 
                      scale=1, arrow_scale=0.5,
                      marker=dict(color='black')
                      )

def cleanup_fig(fig):
    # Make a clean plot with axes etc hidden:
    fig.update_layout(width=800, height=600)
    fig.update_layout(
                coloraxis_showscale=False,
            plot_bgcolor="white",
            paper_bgcolor="white",
            xaxis=dict(
                showgrid=False,
                showticklabels=False,
                showline=False,
                zeroline=False,
                title="",
            ),
            yaxis=dict(
                showgrid=False,
                showticklabels=False,
                showline=False,
                zeroline=False,
                title="",
            ),
            title="",
            margin=dict(l=0, r=0, t=0, b=0),
        )
    fig.update_yaxes(scaleanchor="x", scaleratio=1)

cleanup_fig(fig)
fig.show()

Set up diffusion geometry: construct the exterior derivative d0 for Hodge decomposition.

In [30]:
basis_size = 100

parameters = {}
parameters['n'], parameters['dim'] = x.shape
parameters['n_function_basis'] = basis_size
parameters['n_coefficients'] = basis_size
parameters['projection_tol'] = 1e-2

L, diffuse, lam, u, D = Del0(x, n_function_basis = parameters['n_function_basis'])
data_reg = diffuse @ x

# Compute the weak form of the derivative d0
Gamma_mixed_n0 = Gamma_mixed_n0_pointwise(data_reg, u, lam, L, parameters)
d0_weak = ext_derivative0_weak(u, Gamma_mixed_n0, D, parameters)

Hodge decomposition of the vector field:

In [35]:
# Change the basis of the coefficients to the eigenfunctions.
X = u[:,:parameters['n_coefficients']].T @ (D.reshape(parameters['n'],1) * vf) # [n_coefficients, d]

# Flatten to an [n_coefficients*d] vector.
X = X.flatten()

# Find gradient part of X.
delf = d0_weak.T @ X # [n_function_basis]
factor = 1 / lam
factor[lam < 1e-2] = 0
f = factor * delf
f = u @ f

# We assume f is a linear approximation of the true oxygen values.
# We can use a linear regression to find the best fit parameters.

from sklearn.linear_model import LinearRegression

# Reshape f to be a column vector for sklearn
f_reshaped = f.reshape(-1, 1)  # shape: (N, 1)
true_oxygen_flat = true_oxygen[:-1].flatten()  # shape: (N,)

reg = LinearRegression().fit(f_reshaped, true_oxygen_flat)
fitted_oxygen = reg.predict(f_reshaped)

# Visualise the fitted oxygen
range_color=[1.1 * true_oxygen.min(), 0.9 * true_oxygen.max()]
fig = px.scatter(x=x[:, 0], y=x[:, 1], 
                 color=fitted_oxygen,
                 range_color=range_color)
cleanup_fig(fig)
print("Fitted Oxygen:")
fig.show()

# Visualise the true oxygen
fig = px.scatter(x=x[:, 0], y=x[:, 1], 
                 color=true_oxygen[:-1].flatten(),
                 range_color=[true_oxygen.min(), 0.9 * true_oxygen.max()])
cleanup_fig(fig)
print("True Oxygen:")
fig.show()

Fitted Oxygen:


True Oxygen:
